In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# **Step 1 — Check Kaggle Environment**

In [2]:
# Check Python version

import sys

print("Python version:", sys.version)

Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


# **Step 2 — Install the libraries**

In [3]:
!pip install -q transformers accelerate

# **Step 3 — Check Transformers**

In [4]:
import transformers

print("Transformers version:", transformers.__version__)

Transformers version: 5.0.0


# **Step 4 — Check GPU Availability**

In [5]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.10.0+cpu
GPU available: False


# **Step 5 — Load a Conversational Model**

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

print("Loading model:", MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,  # float32 becuase CPU is using
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Model loaded successfully on:", device)

Loading model: Qwen/Qwen2.5-0.5B-Instruct


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully on: cpu


# **Step 6 — Build the Chatbot Function**

In [7]:
# System prompt — you can change and check chatbot "personality" or can set business use-case
SYSTEM_PROMPT = "You are a helpful, friendly AI assistant for a business. Answer clearly and concisely."

# Conversation history here
conversation_history = [
    {"role": "system", "content": SYSTEM_PROMPT}
]

def chat(user_message, max_new_tokens=200):
    """Ek user message le kar, model se reply generate karta hai aur history update karta hai."""
    conversation_history.append({"role": "user", "content": user_message})

    # Chat template apply and make prompt
    prompt = tokenizer.apply_chat_template(
        conversation_history,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    # only new generated part
    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    reply = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    conversation_history.append({"role": "assistant", "content": reply})
    return reply

# **Step 7 — Test the Chatbot**

In [8]:
response = chat("Hello! Who are you and what can you help me with?")
print("Bot:", response)

Bot: I am an AI language model created by Anthropic to assist users in generating text based on the prompts given to me. I'm here to provide information, answer questions, write stories, and engage in various forms of communication. If there's anything specific you'd like help with or need information about, feel free to ask!


In [9]:
response = chat("Can you give me 3 tips for improving customer service in a small business?")
print("Bot:", response)

Bot: Absolutely! Here are three tips that can significantly improve customer service in a small business:

1. **Customer Feedback Loop**: Implement a clear and accessible feedback system where customers can leave reviews or comments directly through your website or social media pages. This not only helps in addressing issues promptly but also builds trust.

2. **Personalization**: Tailor your interactions with each customer to reflect their unique interests, preferences, and needs. Use data analytics tools to understand customer behavior and tailor services accordingly.

3. **Clear Communication**: Ensure that all staff members receive regular training and updates regarding company policies and procedures. Clear communication ensures that every interaction is handled professionally and efficiently, reducing misunderstandings and frustration among customers.

These strategies can lead to more positive customer experiences, improved satisfaction rates, and higher customer retention.


# **Step 8 — Interactive Chat Loop**

In [10]:
print("Chatbot ready! Type 'exit' to stop.\n")

while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ["exit", "quit"]:
        print("Bot: Goodbye!")
        break
    reply = chat(user_input)
    print("Bot:", reply)

Chatbot ready! Type 'exit' to stop.



You:  hello


Bot: Hello! How can I assist you today?


You:  how can you assist me today?


Bot: As an AI language model, I'm always here to assist you! How may I assist you today?


...

Bot: 当然可以！我今天能为您服务吗？


You:  ok thank you 


Bot: You're welcome! Thank you for using my service. If you have any other questions or need further assistance, feel free to ask.


You:  exit


Bot: Goodbye!
